Creating Assamese Corpus

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

def fetch_assamese_news_content(url):
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    # Extract text from all paragraphs
    text = ' '.join(p.get_text() for p in soup.find_all('p'))
    return text

def split_into_assamese_sentences(text):
    # Split on Assamese danda '।', exclamation, or question mark
    sentences = re.split(r'[।!?]\s*', text)
    return [s.strip() for s in sentences if s.strip()]

def create_assamese_corpus_with_similar_word(url, target_word, output_file='assamese_corpus_similar_word.txt'):
    text = fetch_assamese_news_content(url)
    sentences = split_into_assamese_sentences(text)
    selected_sentences = []
    for sentence in sentences:
        if target_word in sentence:
            selected_sentences.append(sentence)
        if len(selected_sentences) >= 10:
            break
    if not selected_sentences:
        print(f"No sentences found containing the word '{target_word}'")
        return
    with open(output_file, 'w', encoding='utf-8') as f:
        for sentence in selected_sentences:
            f.write(sentence + '।\n')
    print(f"Corpus with sentences containing '{target_word}' written to {output_file}")

# Example usage:
url = "https://www.asomiyapratidin.in/homepage/donald-trump-comments-on-india-us-trade-deal-9473056"
target_word = "আমেৰিকা"  # Change to your desired similar word
create_assamese_corpus_with_similar_word(url, target_word)


Corpus with sentences containing 'আমেৰিকা' written to assamese_corpus_similar_word.txt


In [ ]:
import re
from nltk.util import ngrams

# Read Assamese corpus from file
with open("/content/assamese_corpus_similar_word.txt", "r", encoding="utf-8") as f:
    corpus_text = f.read()

# Tokenize Assamese words using Unicode regex
tokens = re.findall(r'[\u0980-\u09FF]+', corpus_text)

# Generate n-grams
unigrams = list(ngrams(tokens, 1))
bigrams = list(ngrams(tokens, 2))
trigrams = list(ngrams(tokens, 3))

# Display first few n-grams
print("Unigrams:", unigrams[:5])
print("Bigrams:", bigrams[:3])
print("Trigrams:", trigrams[:2])


Unigrams: [('ডিজিটেল',), ('ডেস্কঃ',), ('আমেৰিকা',), ('যুক্তৰাষ্ট্ৰৰ',), ('ৰাষ্ট্ৰপতি',)]
Bigrams: [('ডিজিটেল', 'ডেস্কঃ'), ('ডেস্কঃ', 'আমেৰিকা'), ('আমেৰিকা', 'যুক্তৰাষ্ট্ৰৰ')]
Trigrams: [('ডিজিটেল', 'ডেস্কঃ', 'আমেৰিকা'), ('ডেস্কঃ', 'আমেৰিকা', 'যুক্তৰাষ্ট্ৰৰ')]


In [ ]:
import re
from collections import defaultdict

class NgramModel:
    def __init__(self, n):
        self.n = n
        self.ngram_counts = defaultdict(int)
        self.context_counts = defaultdict(int)
        self.vocabulary = set()

    def tokenize_assamese(self, text):
        # Tokenize Assamese words using Unicode range for Assamese script
        return re.findall(r'[\u0980-\u09FF]+', text)

    def train(self, text):
        tokens = self.tokenize_assamese(text)
        self.vocabulary.update(tokens)
        padded_tokens = ['<s>'] * (self.n-1) + tokens + ['</s>']
        for i in range(len(padded_tokens) - self.n + 1):
            ngram = tuple(padded_tokens[i:i+self.n])
            context = tuple(padded_tokens[i:i+self.n-1])
            self.ngram_counts[ngram] += 1
            self.context_counts[context] += 1

    def probability(self, word, context):
        ngram = context + (word,)
        if self.context_counts[context] > 0:
            return self.ngram_counts[ngram] / self.context_counts[context]
        else:
            return 0.0

    def predict_next_word_probabilities(self, context_word):
        context = (context_word,)
        predictions = {}
        for word in self.vocabulary:
            prob = self.probability(word, context)
            if prob > 0:
                predictions[word] = prob
        # Sort by probability in descending order
        return sorted(predictions.items(), key=lambda x: x[1], reverse=True)

# --- MAIN SCRIPT ---

# Read Assamese corpus from file
with open("/content/assamese_corpus_similar_word.txt", "r", encoding="utf-8") as f:
    corpus_text = f.read()

# Train bigram model
model = NgramModel(2)
model.train(corpus_text)

# Example: Predict next words after a given Assamese word from your corpus
context_word = "ভাৰত"  # Change this to any word from your corpus
predictions = model.predict_next_word_probabilities(context_word)

print(f"Next word probabilities after '{context_word}':")
for word, prob in predictions[:10]:  # Show top 10 predictions
    print(f"P({word} | {context_word}) = {prob:.3f}")


Next word probabilities after 'ভাৰত':
P(আৰু | ভাৰত) = 0.500
P(আমেৰিকাৰ | ভাৰত) = 0.500


Assamese Language Generation Model

In [ ]:
import re
import random
from collections import defaultdict

# Step 1: Read and tokenize the corpus
with open("/content/assamese_corpus_similar_word.txt", "r", encoding="utf-8") as f:
    corpus_text = f.read()

# Split into sentences using Assamese delimiter '।'
sentences = [s.strip() for s in re.split(r'[।!?]\s*', corpus_text) if s.strip()]

# Tokenize each sentence using Assamese Unicode block
def tokenize_assamese(sentence):
    return re.findall(r'[\u0980-\u09FF]+', sentence)

tokenized_sentences = [tokenize_assamese(sentence) for sentence in sentences]

# Step 2: Define the trigram language model
class ToyLanguageModel:
    def __init__(self, n):
        self.n = n
        self.model = defaultdict(lambda: defaultdict(int))
        self.vocab = set()

    def train(self, sentences):
        for sentence in sentences:
            padded_sent = ['<s>'] * (self.n-1) + sentence + ['</s>']
            self.vocab.update(sentence)
            for i in range(len(padded_sent) - self.n + 1):
                context = tuple(padded_sent[i:i+self.n-1])
                word = padded_sent[i+self.n-1]
                self.model[context][word] += 1

    def generate_sentence(self, max_length=15):
        context = tuple(['<s>'] * (self.n-1))
        sentence = []
        for _ in range(max_length):
            possible_words = self.model[context]
            if not possible_words:
                break
            total = sum(possible_words.values())
            probs = {word: count/total for word, count in possible_words.items()}
            word = random.choices(list(probs.keys()), list(probs.values()))[0]
            if word == '</s>':
                break
            sentence.append(word)
            context = tuple(list(context[1:]) + [word])
        return ' '.join(sentence)

# Step 3: Train and generate sentences
model = ToyLanguageModel(3)  # trigram model
model.train(tokenized_sentences)

print("\nGenerated sentences:")
for _ in range(3):
    print(model.generate_sentence())



Generated sentences:
তেওঁ লগতে কয় যে যদি সংশ্লিষ্ট দেশসমূহে কোনো ধৰণৰ শুল্ক ঘোষণা কৰে তেন্তে আমেৰিকাইও শুল্কৰ
ডিজিটেল ডেস্কঃ আমেৰিকা যুক্তৰাষ্ট্ৰৰ ৰাষ্ট্ৰপতি ড নাল্ড ট্ৰাম্পে ভাৰতৰ সৈতে সাম্ভাব্য বাণিজ্যিক চুক্তি নাই
যিবোৰ দেশলৈ ট্ৰাম্পে শুল্ক নীতি সন্দৰ্ভত পত্ৰ প্ৰেৰণ কৰিছে সেইবোৰ দেশৰ আমেৰিকাৰ সৈতে কোনো বাণিজ্যিক


Model Evaluation

In [ ]:
import re

# Read the Assamese corpus file
with open("/content/assamese_corpus_similar_word.txt", "r", encoding="utf-8") as f:
    corpus_text = f.read()

# Split into sentences using Assamese delimiter '।'
sentences = [s.strip() for s in re.split(r'[।!?]\s*', corpus_text) if s.strip()]

# Tokenize each sentence using Assamese Unicode block
def tokenize_assamese(sentence):
    return re.findall(r'[\u0980-\u09FF]+', sentence)

tokenized_sentences = [tokenize_assamese(sentence) for sentence in sentences]


In [ ]:
from collections import defaultdict
import random

class ToyLanguageModel:
    def __init__(self, n):
        self.n = n
        self.model = defaultdict(lambda: defaultdict(int))
        self.vocab = set()

    def train(self, sentences):
        for sentence in sentences:
            padded_sent = ['<s>'] * (self.n-1) + sentence + ['</s>']
            self.vocab.update(sentence)
            for i in range(len(padded_sent) - self.n + 1):
                context = tuple(padded_sent[i:i+self.n-1])
                word = padded_sent[i+self.n-1]
                self.model[context][word] += 1


In [ ]:
import numpy as np

def calculate_perplexity(model, test_sentences):
    total_log_prob = 0
    total_words = 0

    for sentence in test_sentences:
        padded_sent = ['<s>'] * (model.n-1) + sentence + ['</s>']
        for i in range(len(padded_sent) - model.n + 1):
            context = tuple(padded_sent[i:i+model.n-1])
            word = padded_sent[i+model.n-1]
            count = model.model[context][word]
            total = sum(model.model[context].values())
            prob = (count + 1) / (total + len(model.vocab))  # Add-one smoothing
            total_log_prob += np.log2(prob)
            total_words += 1
    return 2 ** (-total_log_prob/total_words)


In [ ]:
# Train trigram model on the Assamese corpus
model = ToyLanguageModel(3)
model.train(tokenized_sentences)

# Use first two sentences as test set for demonstration
test_sentences = tokenized_sentences[:2]

# Calculate perplexity
perplexity = calculate_perplexity(model, test_sentences)
print(f"\nModel perplexity on test set: {perplexity:.2f}")



Model perplexity on test set: 50.30
